## 1.0 Combine the realizations into a single folder

#### -----The following code will append realization files from different folders into a single realization npy file-----

In [1]:
import os
import re
import shutil

def offset_filenames(
    input_dir,
    output_dir,
    prefix="realization_",
    ext=".dat",
    add=3605,
    move=False,
    overwrite=False,
    limit=240,             # how many files to process
    start_index=0,         # <-- NEW: where to start in the sorted list
    case_insensitive=True
):
    """
    Copy/move files like '<prefix><n><ext>' to output_dir as '<prefix><n+add><ext>',
    keeping the SAME prefix. Example: realization_0.dat -> realization_3145.dat
    """
    os.makedirs(output_dir, exist_ok=True)
    flags = re.IGNORECASE if case_insensitive else 0
    rx = re.compile(rf'^{re.escape(prefix)}(\d+){re.escape(ext)}$', flags)

    # Collect matches with their numeric index
    matches = []
    for fname in os.listdir(input_dir):
        m = rx.match(fname)
        if m:
            n = int(m.group(1))
            matches.append((n, fname))

    # Sort by the numeric index so we get a deterministic order
    matches.sort(key=lambda x: x[0])

    processed = 0
    #                 ↓↓↓ start here         ↓↓↓ take `limit` files
    for n, fname in matches[start_index:start_index + limit]:
        new_n = n + add
        src = os.path.join(input_dir, fname)
        dst = os.path.join(output_dir, f"{prefix}{new_n}{ext}")

        if not overwrite and os.path.exists(dst):
            print(f"Skip (exists): {dst}")
            continue

        if move:
            if overwrite and os.path.exists(dst):
                os.remove(dst)
            shutil.move(src, dst)
            print(f"Moved: {fname} → {os.path.basename(dst)}")
        else:
            if overwrite and os.path.exists(dst):
                os.remove(dst)
            shutil.copy2(src, dst)
            print(f"Copied: {fname} → {os.path.basename(dst)}")

        processed += 1

    print(f"Done. Processed {processed} file(s).")


# ====== YOUR PATHS (set OUTSIDE the function) ======
input_dir  = r"C:\Users\xo\Documents\Itasca\flac3d700\My Projects\CNN_Flac3d_3D\2025_Nov19_highsand 100"
output_dir = r"C:\Users\xo\CNN_3D_python\20250805_newcnn model\realization"

offset_filenames(
    input_dir=input_dir,
    output_dir=output_dir,
    prefix="realization_",
    ext=".dat",
    add=3945,        # same offset as before: 60→3205, 299→3444
    move=False,
    overwrite=False,
    limit=100,       # how many files to process
    start_index=0  # skip the first 60 (0–59) that you've already done
)



Copied: realization_0.dat → realization_3945.dat
Copied: realization_1.dat → realization_3946.dat
Copied: realization_2.dat → realization_3947.dat
Copied: realization_3.dat → realization_3948.dat
Copied: realization_4.dat → realization_3949.dat
Copied: realization_5.dat → realization_3950.dat
Copied: realization_6.dat → realization_3951.dat
Copied: realization_7.dat → realization_3952.dat
Copied: realization_8.dat → realization_3953.dat
Copied: realization_9.dat → realization_3954.dat
Copied: realization_10.dat → realization_3955.dat
Copied: realization_11.dat → realization_3956.dat
Copied: realization_12.dat → realization_3957.dat
Copied: realization_13.dat → realization_3958.dat
Copied: realization_14.dat → realization_3959.dat
Copied: realization_15.dat → realization_3960.dat
Copied: realization_16.dat → realization_3961.dat
Copied: realization_17.dat → realization_3962.dat
Copied: realization_18.dat → realization_3963.dat
Copied: realization_19.dat → realization_3964.dat
Copied: re

## 2.0 Prep the combine output folder

#### ----- append the corresponding simulation results into the folders for further process-----

In [3]:
import os
import pandas as pd
import numpy as np

def rename_and_process_from_folder(
    input_dir,
    start_number,
    output_dirs,
    base_names_dict,
    realizations_start=0,
    realizations_count=60,
    increment=0.002,
):
    """
    For each base name pattern in base_names_dict[category], read:
        input_dir / f"{base_name.format(i)}.tab"   for i in [realizations_start, ..., realizations_start+count-1]
    Replace first column with axial strain [0, increment, 2*increment, ...]
    Save to:
        output_dirs[category] / f"{base_name.format(i + start_number)}.csv"
    """
    # Ensure output directories exist
    for out_dir in output_dirs.values():
        os.makedirs(out_dir, exist_ok=True)

    first = realizations_start
    last_excl = realizations_start + realizations_count

    missing = 0
    saved = 0

    for category, base_names in base_names_dict.items():
        for base_name in base_names:
            for realization in range(first, last_excl):
                new_realization = realization + start_number
                input_filename = os.path.join(input_dir, base_name.format(realization) + '.tab')
                output_filename = os.path.join(output_dirs[category], base_name.format(new_realization) + '.csv')

                if not os.path.exists(input_filename):
                    print(f"File not found: {input_filename}")
                    missing += 1
                    continue

                try:
                    # Read whitespace-delimited .tab; let pandas infer header if present
                    df = pd.read_csv(input_filename, sep=r'\s+', engine='python')

                    if df.shape[1] < 2:
                        raise ValueError(f"Expected ≥2 columns in {input_filename}, found {df.shape[1]}.")

                    # Replace first column with axial strain
                    nrows = df.shape[0]
                    new_axial_strain = np.linspace(0.0, (nrows - 1) * increment, nrows)
                    df.iloc[:, 0] = new_axial_strain

                    # Save as CSV
                    df.to_csv(output_filename, index=False)
                    print(f"Saved: {output_filename}")
                    saved += 1

                except Exception as e:
                    print(f"Failed to process {input_filename}: {e}")

    print(f"\nRenaming + processing completed.")
    print(f"  Saved files : {saved}")
    print(f"  Missing files: {missing}")


# === USAGE FOR NEW BATCH: realizations 60–399 → 3605–3944 ===
input_dir = r"C:\Users\xo\Documents\Itasca\flac3d700\My Projects\CNN_Flac3d_3D\2025_Nov19_highsand 100"

# 60 + 3545 = 3605  → realization_60 -> ..._3605
start_number = 3945

output_dirs = {
    'qa':  r"C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input",
    'vol': r"C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input",
    'pp':  r"C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input",
    'ps':  r"C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input",
}

base_names_dict = {
    'qa':  ['ss_z_qa_table_realization_{}',  'ss_y_qa_table_realization_{}',  'ss_x_qa_table_realization_{}'],
    'vol': ['ss_z_vol_table_realization_{}', 'ss_y_vol_table_realization_{}', 'ss_x_vol_table_realization_{}'],
    'pp':  ['ss_z_pp_table_realization_{}',  'ss_y_pp_table_realization_{}',  'ss_x_pp_table_realization_{}'],
    'ps':  ['ss_z_ps_table_realization_{}',  'ss_y_ps_table_realization_{}',  'ss_x_ps_table_realization_{}'],
}

rename_and_process_from_folder(
    input_dir=input_dir,
    start_number=start_number,
    output_dirs=output_dirs,
    base_names_dict=base_names_dict,
    realizations_start=0,      # start at realization 60
    realizations_count=100,     # 60..399 inclusive (399 - 60 + 1 = 340)
    increment=0.002,
)


Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3945.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3946.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3947.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3948.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3949.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3950.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3951.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3952.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_z_qa_table_realization_3953.csv
Saved: C:\

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3967.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3968.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3969.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3970.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3971.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3972.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3973.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3974.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_y_qa_table_realization_3975.csv
Saved: C:\

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3964.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3965.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3966.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3967.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3968.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3969.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3970.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3971.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\soil strength input\ss_x_qa_table_realization_3972.csv
Saved: C:\

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_z_vol_table_realization_3991.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_z_vol_table_realization_3992.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_z_vol_table_realization_3993.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_z_vol_table_realization_3994.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_z_vol_table_realization_3995.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_z_vol_table_realization_3996.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_z_vol_table_realization_3997.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_z_vol_table_realization_3998.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\s

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_y_vol_table_realization_3991.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_y_vol_table_realization_3992.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_y_vol_table_realization_3993.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_y_vol_table_realization_3994.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_y_vol_table_realization_3995.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_y_vol_table_realization_3996.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_y_vol_table_realization_3997.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\ss_y_vol_table_realization_3998.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\volumetric strain input\s

...

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_3993.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_3994.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_3995.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_3996.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_3997.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_3998.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_3999.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_4000.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_y_pp_table_realization_4001.csv
Saved: C:\

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_3994.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_3995.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_3996.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_3997.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_3998.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_3999.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_4000.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_4001.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\pore pressure input\ss_x_pp_table_realization_4002.csv
Saved: C:\

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3977.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3978.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3979.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3980.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3981.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3982.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3983.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3984.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_z_ps_table_realization_3985.csv
S

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3986.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3987.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3988.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3989.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3990.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3991.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3992.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3993.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_y_ps_table_realization_3994.csv
S

Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3986.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3987.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3988.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3989.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3990.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3991.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3992.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3993.csv
Saved: C:\Users\xo\CNN_3D_python\20250805_newcnn model\plastic strain input\ss_x_ps_table_realization_3994.csv
S